In [1]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from scipy.optimize import curve_fit
from scipy.optimize import least_squares

In [ ]:
dataF = np.load('dataF.npy')
dataN = np.load('dataN.npy')
dataO = np.load('dataO.npy')
dataS = np.load('dataS.npy')
dataZ = np.load('dataZ.npy')

folders = [dataZ, dataO, dataN, dataF, dataS]
filenames = ['Z', 'O', 'N', 'F', 'S']

data = dataF[0]

scales = 9
time = np.linspace(0, 23.6, len(dataF[0]))
epochs = np.logspace(
    np.log10(4),
    np.log10(4096/4),
    scales)

x_extrapolate = np.linspace(4, 4096/4, 100)

print(epochs)

In [9]:
def prepros(tseries):
    mean = np.mean(tseries)
    mean_centered = tseries - mean
    prepros = np.cumsum(mean_centered)
    return prepros

def linear(x, m, b):
    return m*x + b

def gaussian(x, amplitude, mean, sigma):
    return amplitude * np.exp(-((x - mean) ** 2) / (2 * sigma ** 2))

def sigmoid(x, L, x0, k, b):
    z = k * (x - x0)
    z = np.clip(z, -60, 60)  # prevent overflow
    return L / (1 + np.exp(-z)) + b

def fitting_sig(x, y, p0=None):
    # default starting guess
    if p0 is None:
        L0 = np.max(y) - np.min(y)
        x0_0 = np.median(x)
        k0 = 1.0
        b0 = np.min(y)
        p0 = [L0, x0_0, k0, b0]

    # residuals
    def residuals(params, x_obs, y_obs):
        return sigmoid(x_obs, *params) - y_obs

    result = least_squares(
        residuals,
        p0,
        args=(x, y),
        method="trf",
        max_nfev=20000
    )

    return result.x

def fitting_lin(segment_x, segment_y):
    popt, pcov = curve_fit(linear, segment_x, segment_y)
    m, b = popt
    return m, b

def RMS_i(f1,f2,n):
    rms_i = np.sqrt( (1/n) * np.sum((f1 - f2)**2) )
    return rms_i

def RMS(residuals):
    F_n = np.sqrt( np.mean (np.array(residuals)**2) )
    return F_n

def DFA(prepros):
    RMS_lin_list = []
    # RMS_sig_list = []

    for i in epochs:
        boundaries = np.arange(0,4097,i).astype(int)
        residuals_lin = np.array([])
        # residuals_sig = np.array([])

        # Epoch processing
        for j in range(len(boundaries)-1):
            segment_x = [k for k in range(int(boundaries[j]), int(boundaries[j+1]))]
            segment_y = prepros[boundaries[j]:boundaries[j+1]]

            # Fitting
            m, b = fitting_lin(segment_x, segment_y)
            # L, x0, k, b = fitting_sig(segment_x, segment_y)

            # RMS Computation 
            np.concatenate(( residuals_lin , np.array(segment_y) - np.array(linear(np.array(segment_x), m, b)) ))
            # np.concatenate(( residuals_sig , np.array(segment_y) - np.array(sigmoid(np.array(segment_x), L, x0, k, b)) ))

        RMS_lin = RMS( residuals_lin )
        RMS_lin_list.append(RMS_lin)
    

    return RMS_lin_list #, RMS_sig_list

In [ ]:
DFA_results = []
DFA_Hurst = []
DFA_linfit = []

processed = prepros(data)
DFA_result = DFA(processed)
DFA_results.append(DFA_result)

L00, x000, k00, b00 = fitting_sig(epochs, DFA_result, p0=[1600,0,0.005,-800])
L01, x001, k01, b01 = fitting_sig(np.log(epochs), DFA_result, p0=[850,200,0.013,-2])
L10, x010, k10, b10 = fitting_sig(epochs, np.log(DFA_result))
L11, x011, k11, b11 = fitting_sig(np.log(epochs), np.log(DFA_result))

print(L01, x001, k01, b01)
# DFA_linfit.append([m1, b1])
# DFA_Hurst.append(m1)

In [ ]:
# PLOTTING

arr = np.random.uniform(-100, 100, size=(1, 4097))[0]
processed2 = prepros(arr)
DFA_result2 = DFA(processed2)
# L100, x1000, k100, b100, y100 = fitting_sig2(DFA_result2, epochs, p0=[1900,-350,0.003,-1400])
# L101, x1001, k101, b101 = fitting_sig2(np.log(epochs), DFA_result2, p0=[4800,1000,0.0004,-1820])
# L110, x1010, k110, b110, y110 = fitting_sig2(np.log(DFA_result2), epochs)
# L111, x1011, k111, b111, y111 = fitting_sig2(np.log(DFA_result2), np.log(epochs))

L100, x1000, k100, b100 = fitting_sig(epochs, DFA_result2, p0=[1900,-350,0.003,-1400])
L101, x1001, k101, b101 = fitting_sig(np.log(epochs), DFA_result2)
L110, x1010, k110, b110 = fitting_sig(epochs, np.log(DFA_result2))
L111, x1011, k111, b111 = fitting_sig(np.log(epochs), np.log(DFA_result2))


fig, axs = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Sigmoid fits for DFA - Log matrix', fontsize=16)
axs[0][0].set_xscale('linear')
axs[0][0].set_yscale('linear')
axs[0][0].set_xlabel('Epoch size (linear)')
axs[0][0].set_ylabel(f'DFA F(n) (linear)')

axs[0][1].set_xscale('log')
axs[0][1].set_yscale('linear')
axs[0][1].set_xlabel('Epoch size (log )')
axs[0][1].set_ylabel('DFA F(n) (linear )')

axs[1][0].set_xscale('linear')
axs[1][0].set_yscale('log')
axs[1][0].set_xlabel('Epoch size (linear scale)')
axs[1][0].set_ylabel('DFA F(n) (log scale)')

axs[1][1].set_xscale('log')
axs[1][1].set_yscale('log')
axs[1][1].set_xlabel('Epoch size (log scale)')
axs[1][1].set_ylabel('DFA F(n) (log scale)')

axs[0][0].scatter(epochs, DFA_result)
axs[0][0].plot(x_extrapolate,sigmoid(x_extrapolate, L00, x000, k00, b00), label=f'EEG:\nL = {L00:.2f}\nx0 = {x000:.2f}\nk = {k00:.3f}\nb = {b00:.2f}', color='blue')
axs[0][0].plot(x_extrapolate,sigmoid(x_extrapolate, L100, x1000, k100, b100), label=f'White Noise:\nL = {L100:.2f}\nx0 = {x1000:.2f}\nk = {k100:.3f}\nb = {b100:.2f}', color='red')
axs[0][0].legend()

axs[0][1].scatter(epochs, DFA_result)
axs[0][1].plot(x_extrapolate,sigmoid(np.log(x_extrapolate), L01, x001, k01, b01), label=f'EEG:\nL = {L01:.2f}\nx0 = {x001:.2f}\nk = {k01:.3f}\nb = {b01:.2f}', color='blue')
axs[0][1].plot(x_extrapolate,sigmoid(np.log(x_extrapolate), L101, x1001, k101, b101), label=f'White Noise:\nL = {L101:.2f}\nx0 = {x1001:.2f}\nk = {k101:.3f}\nb = {b101:.2f}', color='red')
axs[0][1].legend()

axs[1][0].scatter(epochs, DFA_result)
axs[1][0].plot(x_extrapolate,np.exp(sigmoid(x_extrapolate, L10, x010, k10, b10)), label=f'EEG:\nL = {L10:.2f}\nx0 = {x010:.2f}\nk = {k10:.3f}\nb = {b10:.2f}', color='blue')
axs[1][0].plot(x_extrapolate,np.exp(sigmoid(x_extrapolate, L110, x1010, k110, b110)), label=f'White Noise:\nL = {L110:.2f}\nx0 = {x1010:.2f}\nk = {k110:.3f}\nb = {b110:.2f}', color='red')
axs[1][0].legend()

axs[1][1].scatter(epochs, DFA_result)
axs[1][1].plot(x_extrapolate,np.exp(sigmoid(np.log(x_extrapolate), L11, x011, k11, b11)), label=f'EEG:\nL = {L11:.2f}\nx0 = {x011:.2f}\nk = {k11:.3f}\nb = {b11:.2f}', color='blue')
axs[1][1].plot(x_extrapolate,np.exp(sigmoid(np.log(x_extrapolate), L111, x1011, k111, b111)), label=f'White Noise:\nL = {L111:.2f}\nx0 = {x1011:.2f}\nk = {k111:.3f}\nb = {b111:.2f}', color='red')
axs[1][1].legend()


axs[0][0].scatter(epochs, DFA_result2, label=f'White Noise', color='red', alpha=0.5)
axs[0][1].scatter(epochs, DFA_result2, label=f'White Noise', color='red', alpha=0.5)
axs[1][0].scatter(epochs, DFA_result2, label=f'White Noise', color='red', alpha=0.5)
axs[1][1].scatter(epochs, DFA_result2, label=f'White Noise', color='red', alpha=0.5)
axs[1][1].legend()

plt.savefig('Sigmoid fits for DFA - Log matrix.png', dpi=300)



In [ ]:
def prepros(tseries):
    mean = np.mean(tseries)
    mean_centered = tseries - mean
    prepros = np.cumsum(mean_centered)
    return prepros

def linear(x, m, b):
    return m*x + b

def gaussian(x, amplitude, mean, sigma):
    return amplitude * np.exp(-((x - mean) ** 2) / (2 * sigma ** 2))

def sigmoid(x, L, x0, k, b):
    z = k * (x - x0)
    z = np.clip(z, -60, 60)  # prevent overflow
    return L / (1 + np.exp(-z)) + b

def fitting_sig(x, y, p0=None):
    # default starting guess
    if p0 is None:
        L0 = np.max(y) - np.min(y)
        x0_0 = np.median(x)
        k0 = 1.0
        b0 = np.min(y)
        p0 = [L0, x0_0, k0, b0]

    # residuals
    def residuals(params, x_obs, y_obs):
        return sigmoid(x_obs, *params) - y_obs

    result = least_squares(
        residuals,
        p0,
        args=(x, y),
        method="trf",
        max_nfev=20000
    )

    return result.x

def fitting_lin(segment_x, segment_y):
    popt, pcov = curve_fit(linear, segment_x, segment_y)
    m, b = popt
    return m, b

def RMS_i(f1,f2,n):
    rms_i = np.sqrt( (1/n) * np.sum((f1 - f2)**2) )
    return rms_i

def RMS(rms_i_list):
    F_n = np.sqrt( (1/len(rms_i_list)) * np.sum(np.array(rms_i_list)**2) )
    return F_n

def DFA(prepros):
    RMS_lin_list = []
    RMS_sig_list = []

    for i in epochs:
        boundaries = np.arange(0,4097,i).astype(int)
        rms_lin_i_list = []
        rms_sig_i_list = []

        # Epoch processing
        for j in range(len(boundaries)-1):
            segment_x = [k for k in range(int(boundaries[j]), int(boundaries[j+1]))]
            segment_y = prepros[boundaries[j]:boundaries[j+1]]

            # Fitting
            m, b = fitting_lin(segment_x, segment_y)
            L, x0, k, b = fitting_sig(segment_x, segment_y)

            # RMS Computation 
            rms_lin_i = RMS_i(segment_y, linear(np.array(segment_x), m, b), i)
            rms_sig_i = RMS_i(segment_y, sigmoid(np.array(segment_x), L, x0, k, b), i)
            rms_lin_i_list.append(rms_lin_i)
            rms_sig_i_list.append(rms_sig_i)

        F_lin_n = RMS(rms_lin_i_list)
        F_sig_n = RMS(rms_sig_i_list)
        RMS_lin_list.append(F_lin_n)
        RMS_sig_list.append(F_sig_n)
    return RMS_lin_list, RMS_sig_list

In [ ]:
DFA_results = []
DFA_Hurst_lin = []
DFA_Hurst_sig = []

processed = prepros(data)
DFA_result = DFA(processed)
DFA_lin, DFA_sig = DFA_result
m1 , b1 = fitting_lin(DFA_lin, epochs)
m2 , b2 = fitting_lin(DFA_sig, epochs)

DFA_Hurst_lin.append(m1)
DFA_Hurst_sig.append(m2)

In [ ]:
processed = prepros(data)

fig, axs, = plt.subplots(1, 2, figsize=(20, 4))
axs[0].plot(time,data)
axs[1].plot(time,processed)

In [ ]:
fig, axs = plt.subplots( figsize=(10, 10))
# axs.set_xscale('log')
# axs.set_yscale('log')
axs.scatter(epochs, DFA_lin, label='DFA Linear Fit RMS', color='blue')
axs.scatter(epochs, DFA_sig, label='DFA Sigmoid Fit RMS', color='orange')